In [1]:
import io
import pandas as pd
import re

import subprocess

from scipy.stats import pearsonr

In [2]:
def parse_chain_string(chain_data_string):
    
    # Find all full chain entries using a lookahead for 'id:[' and greedily grab until the next 'id:[' or end of string
    chain_entries = re.findall(r'(id:\[.*?)(?=id:\[|$)', chain_data_string, re.DOTALL)

    parsed_chains_list = []

    for entry in chain_entries:
        chain_info = {}
        matches = re.findall(r'([^:]+):\[(.*?)\]', entry, re.DOTALL)
        for key, value in matches:
            chain_info[key] = value
        parsed_chains_list.append(chain_info)

    return parsed_chains_list

# Clean PDA Info

In [10]:

data = pd.read_csv('pda_exported_designs.csv')

training_data = pd.read_csv('../../data/training_data.csv')
training_data['pdb_id'] = training_data['name'].str.extract(r'([0-9][A-Za-z0-9]{3})')
common_pdb_ids = set(data['pdb_code'].str.lower()).intersection(set(training_data['pdb_id'].str.lower()))
cleaned_data = data[~data['pdb_code'].str.lower().isin(common_pdb_ids)].reset_index(drop=True)

sequ_info = cleaned_data[['pdb_code', 'chains','release_date']].copy()
sequ_info['parsed_chains'] = sequ_info['chains'].apply(parse_chain_string)

all_chains_records = []

for index, row in sequ_info.iterrows():
    pdb_code = row['pdb_code'] 
    release_date = row['release_date']
    parsed_chains_for_row = row['parsed_chains']

    if parsed_chains_for_row:
        for chain_dict in parsed_chains_for_row:
            chain_record = {
                'pdb_code': pdb_code,
                'release_date': release_date,
                **chain_dict
            }
            all_chains_records.append(chain_record)
    else:
        all_chains_records.append({
            'pdb_code': pdb_code,
        })

chains_df = pd.DataFrame(all_chains_records)



# Filter PDA Info

In [11]:
denovo_sequences_df = chains_df[
    chains_df['chain_source'].isin(['Synthetic','synthetic construct', 'unknown', 'artificial gene'])
].copy() 


if 'sequence' in denovo_sequences_df.columns:
    denovo_sequences_df['sequence'] = denovo_sequences_df['sequence'].fillna('')

# Filter duplicates
denovo_sequences_df['sequence_length'] = denovo_sequences_df['sequence'].apply(len)
denovo_sequences_df = denovo_sequences_df.drop_duplicates(subset='sequence').reset_index(drop=True)
# print(len(denovo_sequences_df))


# Filter sequences out of range
denovo_sequences_df = denovo_sequences_df[
    (denovo_sequences_df['sequence_length'] >= 40) & 
    (denovo_sequences_df['sequence_length'] <= 250)
].reset_index(drop=True)

print(f"{len(denovo_sequences_df)} sequences of specified length")

# Filter sequences containing parentheses
condition_contains_open_paren = denovo_sequences_df['sequence'].str.contains(r'\(', regex=True)
condition_contains_close_paren = denovo_sequences_df['sequence'].str.contains(r'\)', regex=True)
rows_to_filter_out = (
    condition_contains_open_paren |
    condition_contains_close_paren
)
denovo_sequences_df = denovo_sequences_df[~rows_to_filter_out].reset_index(drop=True)
denovo_sequences_df = denovo_sequences_df.dropna().reset_index(drop=True)

print(f"{len(denovo_sequences_df)} sequences without parentheses")


746 sequences of specified length
529 sequences without parentheses


# Download and clean PDBs


In [19]:
for _, row in denovo_sequences_df.iterrows():
    pdb_code = row['pdb_code']
    chain_id = row['id'][0]
    
    # Command needs Rosetta installed
    command = f"python2.7 $ROSETTA/tools/protein_tools/scripts/clean_pdb.py {pdb_code} {chain_id}"

    try:
        subprocess.run(command, shell=True, check=True)
        print(f"Processed: {pdb_code} with chain {chain_id}")
    except subprocess.CalledProcessError as e:
        print(f"Failed to process {pdb_code} with chain {chain_id}: {e}")

File for 7jzl doesn't exist, downloading from internet.
wget --quiet http://www.rcsb.org/pdb/files/7JZL.pdb.gz -O /home/iwe18/Documents/joh/PhD/projects/denovolikeness/supplements/pda/7JZL.pdb.gz
7JZL E    55 --- --- MOD --- OK
>7JZL_E
DKEWILQKIYEIMRLLDELGHAEASMRVSDLIYEFMKKGDERLLEEAERLLEEVE
Processed: 7jzl with chain E
File for 7zss doesn't exist, downloading from internet.
wget --quiet http://www.rcsb.org/pdb/files/7ZSS.pdb.gz -O /home/iwe18/Documents/joh/PhD/projects/denovolikeness/supplements/pda/7ZSS.pdb.gz
7ZSS D    65 --- --- MOD --- OK
>7ZSS_D
TNMLEALQQRLQFYHGQVARAALENNSGKARRFGRIVKQYEDAIKLYKAGKPVPYDELPVPPGFG
Processed: 7zss with chain D
File for 8dn7 doesn't exist, downloading from internet.
wget --quiet http://www.rcsb.org/pdb/files/8DN7.pdb.gz -O /home/iwe18/Documents/joh/PhD/projects/denovolikeness/supplements/pda/8DN7.pdb.gz
8DN7 D   224 ALT --- MOD --- OK
>8DN7_D
VQLVESGGGLVQPGGSLRLSCAASGFNFYYSSIHWVRQAPGKGLEWVASISSYYGSTSYADSVKGRFTISADTSKNTAYLQMNSLRAEDTAVYYCARETYYTEFSWSYSWGL


gzip: /home/iwe18/Documents/joh/PhD/projects/denovolikeness/supplements/pda/7QNP.pdb.gz: unexpected end of file


7QNP A     0 --- --- --- --- BAD
Processed: 7qnp with chain A
File for 7qnp doesn't exist, downloading from internet.
wget --quiet http://www.rcsb.org/pdb/files/7QNP.pdb.gz -O /home/iwe18/Documents/joh/PhD/projects/denovolikeness/supplements/pda/7QNP.pdb.gz



gzip: /home/iwe18/Documents/joh/PhD/projects/denovolikeness/supplements/pda/7QNP.pdb.gz: unexpected end of file


7QNP B     0 --- --- --- --- BAD
Processed: 7qnp with chain B
File for 7z71 doesn't exist, downloading from internet.
wget --quiet http://www.rcsb.org/pdb/files/7Z71.pdb.gz -O /home/iwe18/Documents/joh/PhD/projects/denovolikeness/supplements/pda/7Z71.pdb.gz
7Z71 B   157 ALT --- MOD --- OK
>7Z71_B
DLGKKLLEAAQIGQLDEVRILMANGADVNASDTDGLTPLHLAAASGHLEIVEVLLKTGADVNATDKWGDTPLHLAASQGHLEIVEVLLKAGADVNATDFTGNTPLHLAAYIGHLEIVEVLLKHGADVNAQDKFGKTPFDLAIDNGNEDIAEVLQKAA
Processed: 7z71 with chain B
File for 8cus doesn't exist, downloading from internet.
wget --quiet http://www.rcsb.org/pdb/files/8CUS.pdb.gz -O /home/iwe18/Documents/joh/PhD/projects/denovolikeness/supplements/pda/8CUS.pdb.gz
8CUS A   135 --- --- MOD --- OK
>8CUS_A
SALAYVMLGLLLSLLNRLSLAAEAYKKAIELDPNDALAWLLLGSVLEKLKRLDEAAEAYKKAIELKPNDASAWKELGKVLEKLGRLDEAAKAYAEAIKLDPSDAEAAKELGKVLEKLGQLELAERAYQLAIELDP
Processed: 8cus with chain A
File for 8cus doesn't exist, downloading from internet.
wget --quiet http://www.rcsb.org/pdb/files/8CUS.pdb.gz -O 

...

7XM1 A     0 --- --- --- --- BAD
Processed: 7xm1 with chain A
File for 8ka6 doesn't exist, downloading from internet.
wget --quiet http://www.rcsb.org/pdb/files/8KA6.pdb.gz -O /home/iwe18/Documents/joh/PhD/projects/denovolikeness/supplements/pda/8KA6.pdb.gz
8KA6 A   144 --- --- MOD --- OK
>8KA6_A
KEELEKLAKELSKVWPELGKLVEEVIKLIEGRSKDPKAAVEGLIETMRRAADLLIEKVLELNPALKDPARTAALVERLLAGEIPSFLSEAGRVLAEAAVAMREAADRLRAELAAGNEDLSAAADEALAVFVEAVRRVAAALLEH
Processed: 8ka6 with chain A
File for 8bl5 doesn't exist, downloading from internet.
wget --quiet http://www.rcsb.org/pdb/files/8BL5.pdb.gz -O /home/iwe18/Documents/joh/PhD/projects/denovolikeness/supplements/pda/8BL5.pdb.gz
8BL5 A   117 --- --- MOD --- OK
>8BL5_A
MLPTAEEVQKLMARYIELMDVGDIEAIVQMYADDATVEAPFGAPPIHGRERIAYFYRRMLGEGIARATLTGPVRASHNGTGAMPFRVEYVFNYAMDVRVEMRFDEHGRIQTMQAYWS
Processed: 8bl5 with chain A
File for 8bl9 doesn't exist, downloading from internet.
wget --quiet http://www.rcsb.org/pdb/files/8BL9.pdb.gz -O /home/iwe18/Documents/joh/PhD/p


gzip: /home/iwe18/Documents/joh/PhD/projects/denovolikeness/supplements/pda/8A3K.pdb.gz: unexpected end of file


8A3K U     0 --- --- --- --- BAD
Processed: 8a3k with chain U
File for 8cyk doesn't exist, downloading from internet.
wget --quiet http://www.rcsb.org/pdb/files/8CYK.pdb.gz -O /home/iwe18/Documents/joh/PhD/projects/denovolikeness/supplements/pda/8CYK.pdb.gz
8CYK B   129 ALT --- MOD --- OK
>8CYK_B
KKLYEYTVTTLDEFLEKLKEFILNTSKDKIYKLTITNPKLIKDIGKAIAKAAEIADVDPKEIEEMIKAVEENELTKLVITIEQTDDKYVIKVELENEDGLVHSFEIYFKNKEEMEKFLELLEKLISKLS
Processed: 8cyk with chain B
File for 2l69 doesn't exist, downloading from internet.
wget --quiet http://www.rcsb.org/pdb/files/2L69.pdb.gz -O /home/iwe18/Documents/joh/PhD/projects/denovolikeness/supplements/pda/2L69.pdb.gz
2L69 A   134 --- --- MOD --- OK
>2L69_A
MNIVIVVFSTDEETLRKFKDIIKKNGFKVRTVRSPQELKDSIEELVKKYNATIVVVVVDDKEWAEKAIRFVKSLGAQVLIIIYDQDQNRLEEFSREVRRRGFEVRTVTSPDDFKKSLERLIREVGSLEHHHHHH
Processed: 2l69 with chain A
File for 2lci doesn't exist, downloading from internet.
wget --quiet http://www.rcsb.org/pdb/files/2LCI.pdb.gz -O /home/iwe18/Documents/joh/PhD


gzip: /home/iwe18/Documents/joh/PhD/projects/denovolikeness/supplements/pda/8F53.pdb.gz: unexpected end of file


8F53 A     0 --- --- --- --- BAD
Processed: 8f53 with chain A
File for 8jvz doesn't exist, downloading from internet.
wget --quiet http://www.rcsb.org/pdb/files/8JVZ.pdb.gz -O /home/iwe18/Documents/joh/PhD/projects/denovolikeness/supplements/pda/8JVZ.pdb.gz
8JVZ A    50 --- --- MOD --- OK
>8JVZ_A
GVKVGDVVEVKKDGKKVVARVVELLHDPARNAPVARVRFEDGEERLILVP
Processed: 8jvz with chain A
File for 2n35 doesn't exist, downloading from internet.
wget --quiet http://www.rcsb.org/pdb/files/2N35.pdb.gz -O /home/iwe18/Documents/joh/PhD/projects/denovolikeness/supplements/pda/2N35.pdb.gz
2N35 A    52 --- --- MOD --- OK
>2N35_A
GTIDEWLLKEAKEKAIEELKKAGITSDYYFDLINKAKTVEGVNALKDEILKA
Processed: 2n35 with chain A
File for 5c39 doesn't exist, downloading from internet.
wget --quiet http://www.rcsb.org/pdb/files/5C39.pdb.gz -O /home/iwe18/Documents/joh/PhD/projects/denovolikeness/supplements/pda/5C39.pdb.gz
5C39 A    51 --- --- MOD --- OK
>5C39_A
DYLRELYKLEQQAMKLYREASEKARNPEKKSVLQKILEDEEKHIEWLETIN
Processed: 5c39 

Relax downloaded structures on cluster:

In [ ]:
#!/bin/bash
#SBATCH --nodes=1
#SBATCH --ntasks=1
#SBATCH --mem=8G
#SBATCH --time=0-30:00:00
#SBATCH --partition=paul
#SBATCH --output=log/log.%J
#SBATCH --job-name="relax"

job=$SLURM_ARRAY_JOB_ID
task=$SLURM_ARRAY_TASK_ID
ROSETTA="/software/all/Rosetta/3.14-gompi-2023b"

module load "Rosetta"

$ROSETTA/bin/relax.mpi.linuxgccrelease -s $1 -nstruct 6 -in:file:fullatom -relax:constrain_relax_to_start_coords -relax:ramp_constraints false -out:prefix relax_

SyntaxError: invalid syntax (1721917383.py, line 11)

# Add Relax Info

In [22]:
with open('relax_score.sc', 'r') as file:
    lines = file.readlines()

filtered_lines = [line.strip() for line in lines if line.strip() and not line.startswith('SEQUENCE:')]

header_line = filtered_lines[0].replace('SCORE:', '').strip()
column_names = [col.strip() for col in header_line.split(' ') if col.strip()]

data_lines = [line.replace('SCORE:', '').strip() for line in filtered_lines[1:] if line.startswith('SCORE: ')]

relax = pd.read_csv(io.StringIO('\n'.join(data_lines)), delim_whitespace=True, header=None, names=column_names)
relax['pdb_code'] = relax['description'].str.split('_').str[1].str.lower()
relax['chain_id'] = relax['description'].str.split('_').str[2]

filtered_relax = relax.loc[relax.groupby(['pdb_code', 'chain_id'])['total_score'].idxmin()].reset_index(drop=True)
filtered_relax['prim_key'] = filtered_relax['pdb_code'] + '_' + filtered_relax['chain_id']
denovo_sequences_df['prim_key'] = denovo_sequences_df['pdb_code'].str.lower() + '_' + denovo_sequences_df['id'].str[0]


merged_relax = pd.merge(denovo_sequences_df, filtered_relax, 
                     left_on=['prim_key'], right_on=['prim_key'], how='inner')

# Add Predictions

Predictions were generated like it is described on github.

In [26]:
predictions = pd.read_csv('pdb_predictions.csv',names=['filename', 'probability'])
predictions['pdb_code'] = predictions['filename'].str.split('_').str[1].str.lower()
predictions['chain_id'] = predictions['filename'].str.split('_').str[2]
predictions['prim_key'] = predictions['pdb_code'] + '_' + predictions['chain_id']
# Add sequence information to the predictions
predictions_merged = pd.merge(predictions, merged_relax, left_on='prim_key', right_on='prim_key', how='inner')
predictions_merged['relative_score'] = predictions_merged['total_score'] / predictions_merged['sequence_length']

# Analyse
Search for wrong predicted de novo proteins

In [27]:
# Probability to be de novo
wrong_classified = predictions_merged[(predictions_merged['probability'] < 0.5)]
print(f"Wrong classified: {len(wrong_classified)}")


# avg per residue score < -0.50
wrong_classified = wrong_classified[wrong_classified['relative_score'] < -0.50]

print(f"Wrong classified with avg per residue score < -0.50: {len(wrong_classified)}")

Wrong classified: 270
Wrong classified with avg per residue score < -0.50: 266


In [34]:
# write fasta file with wrong classified sequences
fasta_string = wrong_classified.apply(
    lambda row: f">{row['prim_key']}\n{row['sequence']}\n", axis=1
).str.cat(sep='')
with open('wrong_classified.fasta', 'w') as fasta_file:
    fasta_file.write(fasta_string)

# Blast
Use the fasta to Blast search and the script extract_json_to_csv.py to convert the blastout json to extracted_blast_results.csv

In [60]:
blast_data = pd.read_csv('extracted_blast_results.csv')
blast_data['pdb_code'] = blast_data['query_title'].str.split('_').str[0].str.lower()
blast_data['chain_id'] = blast_data['query_title'].str.split('_').str[1].str.split(',').str[0]
blast_data['prim_key'] = blast_data['pdb_code'] + '_' + blast_data['chain_id']

In [ ]:
filter = blast_data[(blast_data['percent_id'] < 100)].reset_index(drop=True)
print(f"Proteins with blast hits (excluded 100% identity): {len(filter['prim_key'].unique())}")

# Proteins that are de novo by our definition but got not classified as such by our classifier
missed_proteins = wrong_classified[~wrong_classified['prim_key'].isin(filter['prim_key'])]

print(f"Missed proteins: {len(missed_proteins)}")

Proteins with blast hits (excluded 100% identity): 226
Missed proteins: 40


In [52]:


years = missed_proteins['release_date'].apply(lambda x: int(x.split('-')[0])).value_counts().sort_index()
counts = years.values
print(f"Years: {years.index.values}")
print(f"Counts: {counts}")

# Pearson correlation coefficient
corr_coef, p_value = pearsonr(years.index.values, counts)

print(f"Pearson correlation coefficient: {corr_coef:.3f}")
print(f"P-value: {p_value:.3g}")

if p_value < 0.05:
    print("There is a statistically significant correlation between release year and number of misclassified PDB structures.")
else:
    print("There is no statistically significant correlation between release year and number of misclassified PDB structures.")

Years: [2016 2017 2018 2020 2021 2022 2023 2024]
Counts: [ 2  2  7  1  2  6  6 11]
Pearson correlation coefficient: 0.617
P-value: 0.103
There is no statistically significant correlation between release year and number of misclassified PDB structures.
